In [1]:
# Scraped from https://en.wikipedia.org/wiki/Llama
with open("./data/Llama_Wikipedia_Cleaned.txt", "r", encoding="utf-8") as file:
    content = file.read()

In [2]:
# Function to split content into chunks of equal length
def split_content_into_equal_length(content, char_length):
    return [content[i:i+char_length] for i in range(0, len(content), char_length)]

split_content_into_equal_length(content, 300)[:3]

['Llama - Wikipedia > Excerpt > Not to be confused with Ilama or Lama. --- Not to be confused with Ilama "Ilama (disambiguation)") or Lama. Eukaryota Kingdom: Chordata Class: Artiodactyla Family: Lama "Lama (genus)") Species: The llama (; Spanish pronunciation: [\\[ˈʎama\\]]( "Help:IPA/Spanish") or \\[ˈʝ',
 'ama\\]) (Lama glama) is a domesticated South American camelid, widely used as a meat and pack animal by Andean cultures since the pre-Columbian era. Llamas are social animals and live with others as a herd. Their wool is soft and contains only a small amount of lanolin.[2] Llamas can learn simple tas',
 'ks after a few repetitions. When using a pack, they can carry about 25 to 30% of their body weight for 8 to 13 km (5–8 miles).[3] The name llama (in the past also spelled "lama" or "glama") was adopted by European settlers from native Peruvians.[4] The ancestors of llamas are thought to have origina']

In [3]:
def simple_sentence_splitter(text, chunk_size=5):
    # Split text based on a period followed by a space, which is a common end of sentence marker
    sentences = text.split('. ')
    chunks = []

    # Group sentences into chunks of 'chunk_size'
    for i in range(0, len(sentences), chunk_size):
        chunk = '. '.join(sentences[i:i+chunk_size]) + '.'
        chunks.append(chunk)

    return chunks

# Example usage with the 'content' variable
content_chunks = simple_sentence_splitter(content, 5)
print(content_chunks[:3])

['Llama - Wikipedia > Excerpt > Not to be confused with Ilama or Lama. --- Not to be confused with Ilama "Ilama (disambiguation)") or Lama. Eukaryota Kingdom: Chordata Class: Artiodactyla Family: Lama "Lama (genus)") Species: The llama (; Spanish pronunciation: [\\[ˈʎama\\]]( "Help:IPA/Spanish") or \\[ˈʝama\\]) (Lama glama) is a domesticated South American camelid, widely used as a meat and pack animal by Andean cultures since the pre-Columbian era. Llamas are social animals and live with others as a herd. Their wool is soft and contains only a small amount of lanolin.[2] Llamas can learn simple tasks after a few repetitions.', 'When using a pack, they can carry about 25 to 30% of their body weight for 8 to 13 km (5–8 miles).[3] The name llama (in the past also spelled "lama" or "glama") was adopted by European settlers from native Peruvians.[4] The ancestors of llamas are thought to have originated from the Great Plains of North America about 40 million years ago, and subsequently mig

In [4]:
# Import the RecursiveCharacterTextSplitter from langchain for splitting text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
# Initialize the RecursiveCharacterTextSplitter with desired parameters
# This will allow us to split the content into manageable chunks for further processing
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", "? ", "! "],  # List of characters to split on
    chunk_size=600,  # The maximum size of your chunks
    chunk_overlap=50,  # The maximum overlap between chunks
)

In [6]:
langchain_chunks = text_splitter.create_documents([content])
langchain_chunks[:1]

[Document(metadata={}, page_content='Llama - Wikipedia > Excerpt > Not to be confused with Ilama or Lama. --- Not to be confused with Ilama "Ilama (disambiguation)") or Lama. Eukaryota Kingdom: Chordata Class: Artiodactyla Family: Lama "Lama (genus)") Species: The llama (; Spanish pronunciation: [\\[ˈʎama\\]]( "Help:IPA/Spanish") or \\[ˈʝama\\]) (Lama glama) is a domesticated South American camelid, widely used as a meat and pack animal by Andean cultures since the pre-Columbian era. Llamas are social animals and live with others as a herd')]

In [7]:
# Create a text splitter with only chunk size and overlap specified
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,  # The maximum size of your chunks
    chunk_overlap=300,  # The maximum overlap between chunks
)

## YOUR SOLUTION HERE ##
# Split the 'content' into chunks
langchain_chunks = text_splitter.create_documents([content])

## YOUR SOLUTION HERE ##
# Print the first document
langchain_chunks[:1]

[Document(metadata={}, page_content='Llama - Wikipedia > Excerpt > Not to be confused with Ilama or Lama. --- Not to be confused with Ilama "Ilama (disambiguation)") or Lama. Eukaryota Kingdom: Chordata Class: Artiodactyla Family: Lama "Lama (genus)") Species: The llama (; Spanish pronunciation: [\\[ˈʎama\\]]( "Help:IPA/Spanish") or \\[ˈʝama\\]) (Lama glama) is a domesticated South American camelid, widely used as a meat and pack animal by Andean cultures since the pre-Columbian era. Llamas are social animals and live with others as a herd. Their wool is soft and contains only a small amount of lanolin.[2] Llamas can learn simple tasks after a few repetitions. When using a pack, they can carry about 25 to 30% of their body weight for 8 to 13 km (5–8 miles).[3] The name llama (in the past also spelled "lama" or "glama") was adopted by European settlers from native Peruvians.[4] The ancestors of llamas are thought to have originated from the Great Plains of North America about 40 million

In [8]:
import chromadb

# Instantiate a Chroma client to interact with the Chroma database
chroma_client = chromadb.Client()

# Create a new collection named "llama_chunks"
if "llama_chunks" not in [col.name for col in chroma_client.list_collections()]:
    collection = chroma_client.create_collection(name="llama_chunks")
else:
    collection = chroma_client.get_collection(name="llama_chunks")

# Iterate over the langchain_chunks using a for loop and enumerate
for index, chunk in enumerate(langchain_chunks):
    # Add a new document to the collection
    collection.add(
        ids=[f"chunk_{index}"],  
        documents=[chunk.page_content],
        metadatas=[{"source": "https://en.wikipedia.org/wiki/Llama",
                    "chunk_index": index}] 
    )

results = collection.query(query_texts=["Are llamas domesticated?"], n_results=1)

print(results)

{'ids': [['chunk_1']], 'embeddings': None, 'documents': [['were over seven million llamas and alpacas in South America and over 158,000 llamas and 100,000 alpacas, descended from progenitors imported late in the 20th century, in the United States and Canada.[5] In Aymara mythology, llamas are important beings. The Heavenly Llama is said to drink water from the ocean and urinates as it rains.[6] According to Aymara eschatology, llamas will return to the water springs and ponds where they come from at the end of time.[6] Classification , consist of the vicuña (Vicugna vicugna, prev. Lama vicugna), guanaco (Lama guanicoe), Suri alpaca, and Huacaya alpaca (Vicugna pacos, prev. Lama guanicoe pacos), and the domestic llama (Lama glama). Guanacos and vicuñas live in the wild, while llamas and alpacas exist only as domesticated animals.[7] Although early writers compared llamas to sheep, their similarity to the camel was soon recognized. They were included in the genus Camelus along with alpac